# Colab TTS Quality Gate

Run on a Google Colab GPU runtime (T4 is sufficient). This notebook tests REAL Qwen3-TTS and REAL Chatterbox-Turbo; it never creates placeholder audio.

Outputs: `projects/colab-tts-quality/qwen3_tts_output.wav`, `chatterbox_output.wav`, and `tts_quality_report.json`.


In [ ]:
import os, sys, subprocess, importlib.util
print('Python:', sys.version)
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('PyTorch is missing. Start a Colab GPU runtime and run this notebook from the top.')
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. In Colab choose Runtime > Change runtime type > T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
# Install repo TTS requirements without replacing Colab's working CUDA PyTorch.
# The old notebook installed a hard-coded cu118 torch wheel, which can break a newer Colab runtime.
import os, subprocess, sys
REPO = '/content/openmontage-colab'
if not os.path.isdir(REPO):
    !git clone -b tga892338-rgb-refactor-colab-ready https://github.com/tga892338-rgb/openmontage-colab.git {REPO}
os.chdir(REPO)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-tts.txt'])
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg'], check=True)
print('Dependencies installed.')
import torch
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'available:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('CUDA disappeared after installation; restart the runtime and rerun from the top.')


In [ ]:
from pathlib import Path
import json, os, subprocess, time, traceback
import torch
out_dir = Path('/content/openmontage-colab/projects/colab-tts-quality')
out_dir.mkdir(parents=True, exist_ok=True)
qwen_path = out_dir / 'qwen3_tts_output.wav'
chatter_path = out_dir / 'chatterbox_output.wav'
report_path = out_dir / 'tts_quality_report.json'
text = ('A hundred years ago, humanity looked toward the stars and wondered whether we were alone.\n\n'
        'Tonight, something answered.\n\n'
        'The signal came from a world no telescope had ever seen before.\n\n'
        'And buried inside that transmission was a message meant for us.')
instruction = ('Calm cinematic documentary narration. Natural pacing. Clear pronunciation. '
               'Slight sense of mystery and anticipation. Subtle emotional expression. '
               'Avoid sounding like an advertisement or an AI assistant.')
def ffprobe_info(path):
    p = subprocess.run(['ffprobe','-v','error','-show_format','-show_streams','-print_format','json',str(path)], capture_output=True, text=True)
    return (json.loads(p.stdout), None) if p.returncode == 0 else (None, p.stderr)
def file_ok(path):
    if not path.exists() or path.stat().st_size == 0: return False, 'missing-or-zero-size'
    info, err = ffprobe_info(path)
    return (True, info) if info else (False, f'ffprobe-failed: {err}')
def vram_snapshot():
    torch.cuda.synchronize()
    return {'allocated_mb': round(torch.cuda.memory_allocated()/2**20,2), 'reserved_mb': round(torch.cuda.memory_reserved()/2**20,2), 'max_allocated_mb': round(torch.cuda.max_memory_allocated()/2**20,2)}


## Qwen3-TTS 0.6B — REAL test
Uses the current `qwen-tts` package/API. A failure is recorded rather than silently producing fake audio.

In [ ]:
qwen_result={'model':'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice','status':'NOT_RUN'}
try:
    import torch, time
    from qwen_tts import Qwen3TTS
    torch.cuda.reset_peak_memory_stats()
    t0=time.time()
    model=Qwen3TTS('Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice', device_map='cuda:0', dtype=torch.bfloat16)
    qwen_result['model_load_time']=time.time()-t0
    qwen_result['vram_after_load']=vram_snapshot()
    t0=time.time()
    # Official CustomVoice interface returns a list of (waveform, sample_rate).
    wavs = model.generate_custom_voice(text=text, language='English', speaker='Ryan', instruct=instruction)
    qwen_result['gen_time']=time.time()-t0
    wav, sr = wavs[0]
    import soundfile as sf, numpy as np
    wav=np.asarray(wav)
    sf.write(str(qwen_path), wav, int(sr))
    ok, info=file_ok(qwen_path)
    qwen_result.update({'sample_rate':int(sr),'file_ok':ok,'ffprobe':info,'status':'REAL_PASS' if ok else 'REAL_FAIL','vram_after_gen':vram_snapshot()})
    print('QWEN3 REAL =', qwen_result['status'])
except Exception as e:
    qwen_result.update({'status':'REAL_FAIL','error':str(e),'traceback':traceback.format_exc()})
    print('QWEN3 REAL = FAIL\n', qwen_result['traceback'])


## Chatterbox-Turbo 350M — REAL test
Uses the current `chatterbox-tts` package/API. No placeholder bytes are accepted.

In [ ]:
chatter_result={'model':'ResembleAI/chatterbox-turbo','status':'NOT_RUN'}
try:
    import torch, time
    from chatterbox.tts_turbo import ChatterboxTurboTTS
    torch.cuda.reset_peak_memory_stats()
    t0=time.time()
    model2=ChatterboxTurboTTS.from_pretrained(device='cuda')
    chatter_result['model_load_time']=time.time()-t0
    chatter_result['vram_after_load']=vram_snapshot()
    t0=time.time()
    wav=model2.generate(text)
    chatter_result['gen_time']=time.time()-t0
    import torchaudio
    torchaudio.save(str(chatter_path), wav.cpu(), model2.sr)
    ok, info=file_ok(chatter_path)
    chatter_result.update({'sample_rate':int(model2.sr),'file_ok':ok,'ffprobe':info,'status':'REAL_PASS' if ok else 'REAL_FAIL','vram_after_gen':vram_snapshot()})
    print('CHATTERBOX REAL =', chatter_result['status'])
except Exception as e:
    chatter_result.update({'status':'REAL_FAIL','error':str(e),'traceback':traceback.format_exc()})
    print('CHATTERBOX REAL = FAIL\n', chatter_result['traceback'])


In [ ]:
from IPython.display import Audio, display
report={'gpu':{'name':torch.cuda.get_device_name(0),'torch':torch.__version__,'cuda':torch.version.cuda},'qwen':qwen_result,'chatterbox':chatter_result}
with open(report_path,'w',encoding='utf-8') as f: json.dump(report,f,indent=2,default=str)
print(json.dumps(report,indent=2,default=str))
if qwen_path.exists() and qwen_result.get('status')=='REAL_PASS': display(Audio(str(qwen_path)))
if chatter_path.exists() and chatter_result.get('status')=='REAL_PASS': display(Audio(str(chatter_path)))
print('Report:', report_path)
